In [6]:
import sys

def print_board(state):
    """Prints the 3x3 board in a grid format."""
    for i in range(0, 9, 3):
        print(f" {state[i]} {state[i+1]} {state[i+2]} ")
    print()

def get_neighbors(state):
    """Generates all valid next states by moving the blank tile (0)."""
    neighbors = []
    blank_idx = state.index(0)
    row, col = blank_idx // 3, blank_idx % 3

    # Define possible moves for the blank tile: (row_change, col_change)
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for dr, dc in moves:
        new_row, new_col = row + dr, col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_blank_idx = new_row * 3 + new_col
            new_state = list(state)
            new_state[blank_idx], new_state[new_blank_idx] = new_state[new_blank_idx], new_state[blank_idx]
            neighbors.append(tuple(new_state))

    return neighbors

def depth_limited_dfs(state, goal_state, limit, path, visited, stats):
    """Performs DFS up to a specific depth limit while tracking metrics."""
    # Increment total states explored (Time Complexity tracker)
    stats['nodes_explored'] += 1

    # Track maximum recursion depth active at one time (Space Complexity tracker)
    current_depth = len(path)
    if current_depth > stats['max_stack_depth']:
        stats['max_stack_depth'] = current_depth

    if state == goal_state:
        return path + [state]
    if limit <= 0:
        return None

    visited.add(state)
    # Track the peak unique states stored globally across this depth iteration
    if len(visited) > stats['max_visited_size']:
        stats['max_visited_size'] = len(visited)

    for neighbor in get_neighbors(state):
        if neighbor not in visited:
            result = depth_limited_dfs(neighbor, goal_state, limit - 1, path + [state], visited, stats)
            if result is not None:
                return result

    visited.remove(state)
    return None

def solve_iddfs(start_state, goal_state, max_depth=32):
    """Gradually increases depth limit to guarantee the shortest path."""
    # System statistics to calculate theoretical and actual complexity
    stats = {
        'nodes_explored': 0,
        'max_stack_depth': 0,
        'max_visited_size': 0
    }

    for depth in range(max_depth + 1):
        visited = set()
        result = depth_limited_dfs(start_state, goal_state, depth, [], visited, stats)
        if result is not None:
            return result, stats

    return None, stats

def is_solvable(state):
    """Checks if an 8-puzzle configuration is mathematically solvable."""
    flat_list = [tile for tile in state if tile != 0]
    inversions = 0
    for i in range(len(flat_list)):
        for j in range(i + 1, len(flat_list)):
            if flat_list[i] > flat_list[j]:
                inversions += 1
    return inversions % 2 == 0

def main():
    goal_state = (1, 2, 3, 4, 5, 6, 7, 8, 0)
    print("1WN24CS062")
    print("BHASHITHA G S")
    print("--- Optimal 8-Puzzle Visual Solver (with Complexity Analysis) ---")
    print("Enter the initial board configuration row by row.")
    print("Use numbers 1-8 for tiles, and 0 for the empty space.")
    print("Example input: 1 2 3 4 0 5 6 7 8\n")

    try:
        user_input = input("Enter 9 space-separated numbers: ").strip().split()
        if len(user_input) != 9:
            print("Error: You must enter exactly 9 numbers.")
            return

        start_state = tuple(int(x) for x in user_input)
        if sorted(start_state) != list(range(9)):
            print("Error: Input must contain numbers from 0 to 8 with no duplicates.")
            return

    except ValueError:
        print("Error: Invalid input. Please enter numbers only.")
        return

    if not is_solvable(start_state):
        print("\nInitial Board:")
        print_board(start_state)
        print("This specific puzzle state is mathematically UNSOLVABLE.")
        return

    print("\nSearching for the optimal solution...")
    solution_path, complexity_stats = solve_iddfs(start_state, goal_state)

    if solution_path is not None:
        num_moves = len(solution_path) - 1

        print(f"\n======================================")
        print(f"        COMPLEXITY ANALYSIS           ")
        print(f"======================================")
        print(f"• TIME COMPLEXITY:")
        print(f"  - Theoretical Bound: O(b^d) where b (branching factor) ≈ 3, d (depth) = {num_moves}")
        print(f"  - Actual Execution: Explored {complexity_stats['nodes_explored']} total states.")
        print(f"• SPACE COMPLEXITY:")
        print(f"  - Theoretical Bound: O(d) linear memory required for the path.")
        print(f"  - Max Recursion Stack Depth: {complexity_stats['max_stack_depth']} frames frame structures.")
        print(f"  - Max Memory Allocations: Tracked {complexity_stats['max_visited_size']} unique concurrent states.")
        print(f"======================================\n")

        print(f"Optimal solution found in {num_moves} moves!\n")
        for step, board_state in enumerate(solution_path):
            if step == 0:
                print("Initial State:")
            elif step == len(solution_path) - 1:
                print(f"Step {step} (Goal Reached):")
            else:
                print(f"Step {step}:")
            print_board(board_state)
    else:
        print("No solution found within the maximum depth limit.")

if __name__ == "__main__":
    main()


1WN24CS062
BHASHITHA G S
--- Optimal 8-Puzzle Visual Solver (with Complexity Analysis) ---
Enter the initial board configuration row by row.
Use numbers 1-8 for tiles, and 0 for the empty space.
Example input: 1 2 3 4 0 5 6 7 8

Enter 9 space-separated numbers: 1 2 3 0 4 6 7 5 8

Searching for the optimal solution...

        COMPLEXITY ANALYSIS           
• TIME COMPLEXITY:
  - Theoretical Bound: O(b^d) where b (branching factor) ≈ 3, d (depth) = 3
  - Actual Execution: Explored 30 total states.
• SPACE COMPLEXITY:
  - Theoretical Bound: O(d) linear memory required for the path.
  - Max Recursion Stack Depth: 3 frames frame structures.
  - Max Memory Allocations: Tracked 3 unique concurrent states.

Optimal solution found in 3 moves!

Initial State:
 1 2 3 
 0 4 6 
 7 5 8 

Step 1:
 1 2 3 
 4 0 6 
 7 5 8 

Step 2:
 1 2 3 
 4 5 6 
 7 0 8 

Step 3 (Goal Reached):
 1 2 3 
 4 5 6 
 7 8 0 

